# 01 · Classical Models — Axis × Position × LOSO

**Project:** Sensor-Efficient Wearable Gym Exercise Recognition: An Axis-Level and Sensor-Placement Analysis Under Subject-Independent Evaluation

Runs **Random Forest, SVM and XGBoost** on every axis configuration (15) at every position (3) with **Leave-One-Subject-Out** cross-validation. It reports nothing until *you* run it: all numbers come from the real file on your Colab/Kaggle run.

### Design decisions frozen **before** any model was run
| Decision | Value | Reason |
|---|---|---|
| Channels | Ax Ay Az Gx Gy Gz (C1 excluded) | Primary-study protocol |
| Window | 4 s = 80 samples @ 20 Hz, 50 % overlap | Chosen a priori (long enough to span roughly one repetition of slower lifts); audit showed it is feasible for every subject × class × position. **Not tuned on any test result.** Change only in the config cell and re-run from scratch. |
| Window purity | A window never crosses a (subject, position, session, label) boundary | Prevents label mixing and session bleeding |
| Classes | All 12 actual labels incl. `Null` | Primary task = dataset's own label set |
| Primary metric | Macro-F1 over classes present in the held-out subject | |
| Secondary | Accuracy, weighted-F1, macro-precision, macro-recall, **macro-F1 over the 11 workouts only** (same predictions, `Null` excluded from the average), per-class F1 (for RQ4) | No extra training needed |
| Features | 15 per-channel features (time + FFT). **No cross-channel features** → a configuration with k channels sees exactly 15·k features | Keeps the axis ablation clean; checked by a unit test below |
| Class imbalance | Class weights computed on training windows only | `Null` is an extreme class (audit) |
| Hyper-parameters | **Fixed, pre-declared** (see config). No tuning → no test-subject influence by construction | Nested subject-grouped tuning can be added later as a separate, clearly labelled step |
| Scaling | `StandardScaler` fitted on training windows of each fold only (SVM); trees are scale-invariant | |

### Leakage guards implemented
- LOSO folds are by **subject**; asserts in every fold that the test subject is absent from the training windows.
- Feature extraction is per window (no cross-window statistics).
- Scaler, class weights and SVM sub-sampling use training windows only.
- Windows are built before any fold exists, but they contain no label-dependent or subject-dependent normalisation.
- Caveat to state in the paper: the dataset itself is *already normalised by its providers* (method undocumented), which this pipeline cannot undo.

### Resumable
Every (position, fold, config, model) unit is appended to `results.csv` immediately. If Colab disconnects, re-run all: finished units are skipped. A config-hash check refuses to mix results from different settings.

> **First run with `SMOKE_TEST = True`** (default) to verify everything and see timing; then set `SMOKE_TEST = False` for the full grid. Smoke results go in a separate folder and are never mixed with the full run.

## 1 · Configuration

In [ ]:
import os, sys, re, json, glob, time, hashlib, platform, random, shutil, zipfile, warnings, itertools, io
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ============================ USER CONFIG ====================================
SMOKE_TEST = bool(os.environ.get("SMOKE_TEST", "1") == "1")   # True = tiny check run; False = full grid
SEED = 42
DATA_PATH = os.environ.get("RECGYM_CSV", "data/RecGym.csv")
RESULTS_ROOT = os.environ.get("RESULTS_DIR", "results/raw/classical")   # parent dir; "full" or "smoke" is appended below
EXPECTED_SHA256 = "2031c077a57e93d6b5c39da198470cb6c9b1a3c0652ab57cfc5d6a39bfa840e9"   # from your audit
ENFORCE_HASH = os.environ.get("RECGYM_SKIP_HASH") is None
MOUNT_DRIVE = True
# (RESULTS_ROOT is set from the RESULTS_DIR environment variable above.)

FS = 20                      # nominal sampling rate (documentation)
WINDOW_SECONDS = 4
OVERLAP = 0.5
MODELS = ["RF", "SVM", "XGB"]
POSITIONS = None             # None = all; or e.g. ["wrist"]
CONFIG_NAMES = None          # None = all 15; or e.g. ["6-axis", "acc-3"]
FOLDS = None                 # None = all LOSO folds; or list of fold indices

RF_PARAMS = dict(n_estimators=200, max_depth=None, min_samples_leaf=1, max_features="sqrt",
                 class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
SVM_PARAMS = dict(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", cache_size=1000, random_state=SEED)
SVM_MAX_TRAIN_WINDOWS = 15000        # stratified sub-sample of TRAINING windows only (SVM is O(n^2))
XGB_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                  tree_method="hist", n_jobs=-1, random_state=SEED)
LATENCY_N_WINDOWS = 50
FORCE_NEW_RUN = False        # True = ignore a config-hash mismatch and overwrite run_config.json (results.csv is NOT deleted)
# ==============================================================================

if SMOKE_TEST:
    POSITIONS = POSITIONS or None
    CONFIG_NAMES = CONFIG_NAMES or ["6-axis", "single-Ax"]
    FOLDS = FOLDS or [0, 1]
    RF_PARAMS = {**RF_PARAMS, "n_estimators": 30}
    XGB_PARAMS = {**XGB_PARAMS, "n_estimators": 30}
    SVM_MAX_TRAIN_WINDOWS = 4000

IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE = Path("/kaggle/working") if IN_KAGGLE else (Path("/content") if IN_COLAB else Path(".").resolve())
random.seed(SEED); np.random.seed(SEED)

if IN_COLAB and MOUNT_DRIVE and str(DATA_PATH).startswith("/content/drive") and not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

RESULTS_ROOT = Path(RESULTS_ROOT)
RUN_DIR = RESULTS_ROOT / ("smoke" if SMOKE_TEST else "full")
SHARED = RESULTS_ROOT / "shared"
for d in [RUN_DIR, SHARED]:
    d.mkdir(parents=True, exist_ok=True)

LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

log("SMOKE_TEST =", SMOKE_TEST, "| results dir:", RUN_DIR)
if SMOKE_TEST:
    log(">>> SMOKE TEST: reduced grid and reduced model sizes. Set SMOKE_TEST = False for the real run. <<<")

## 2 · Load data and verify it is the audited file

In [ ]:
def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

assert os.path.exists(DATA_PATH), f"Data file not found: {DATA_PATH}. Mount Drive or set DATA_PATH."
DATA_SHA256 = sha256_of(DATA_PATH)
log("SHA-256:", DATA_SHA256)
if ENFORCE_HASH:
    assert DATA_SHA256 == EXPECTED_SHA256, ("The file differs from the one audited in notebook 00. "
        "Re-run the audit on this file (or update EXPECTED_SHA256 on purpose).")
    log("Hash matches the audited file.")

raw = pd.read_csv(DATA_PATH)
raw.columns = [str(c).strip() for c in raw.columns]
log("Loaded:", raw.shape, list(raw.columns))

def norm(c): return re.sub(r"[^a-z0-9]", "", str(c).lower())
ncol = {norm(c): c for c in raw.columns}
def pick(cands):
    for k in cands:
        if k in ncol: return ncol[k]
    return None
subj_col = pick(["subject", "object", "subjectid", "volunteer", "participant", "user"])
label_col = pick(["workout", "label", "activity", "exercise"])
pos_col = pick(["position", "placement", "location"])
sess_col = pick(["session", "sessionid", "day"])
IMU = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
chan_cols = {k: ncol.get(k.lower()[0] + "_" + k.lower()[1]) or ncol.get(k.lower()) for k in IMU}
assert subj_col and label_col and pos_col, f"Cannot detect roles in {list(raw.columns)}"
assert all(chan_cols.values()), f"Cannot detect IMU channels: {chan_cols}"
log("Roles:", dict(subject=subj_col, label=label_col, position=pos_col, session=sess_col), "| channels:", chan_cols)
if sess_col is None:
    log("WARNING: no session column; runs are delimited by (subject, position, label) only.")

subject_str = raw[subj_col].astype(str).str.strip()
subjects = sorted(subject_str.unique(), key=lambda s: (len(s), s))
labels = sorted(raw[label_col].astype(str).str.strip().unique())
positions_all = sorted(raw[pos_col].astype(str).str.strip().unique())
K = len(labels)
label_map = {l: i for i, l in enumerate(labels)}
null_ids = [label_map[l] for l in labels if "null" in l.lower()]
log("Subjects:", subjects); log("Positions:", positions_all); log("Labels:", labels, "| null ids:", null_ids)

cat = lambda s, cats: pd.Categorical(s.astype(str).str.strip(), categories=cats).codes.astype(np.int32)
subj_c = cat(raw[subj_col], subjects)
pos_c = cat(raw[pos_col], positions_all)
lab_c = cat(raw[label_col], labels)
sess_vals = sorted(raw[sess_col].astype(str).str.strip().unique()) if sess_col else ["NA"]
sess_c = cat(raw[sess_col], sess_vals) if sess_col else np.zeros(len(raw), dtype=np.int32)
assert (subj_c >= 0).all() and (pos_c >= 0).all() and (lab_c >= 0).all()
sig = raw[[chan_cols[k] for k in IMU]].to_numpy(dtype=np.float32)
assert np.isfinite(sig).all(), "non-finite IMU values: handle explicitly before modelling"
n_rows = len(raw); del raw
log("Sessions:", sess_vals)

## 3 · Build windows (inside runs only) and save the window index
The index is saved so the later 1D-CNN notebook reuses the **identical** windows.

In [ ]:
W = int(round(WINDOW_SECONDS * FS)); S = max(1, int(round(W * (1 - OVERLAP))))
chg = np.ones(n_rows, dtype=bool)
chg[1:] = ((subj_c[1:] != subj_c[:-1]) | (pos_c[1:] != pos_c[:-1]) |
           (sess_c[1:] != sess_c[:-1]) | (lab_c[1:] != lab_c[:-1]))
starts = np.flatnonzero(chg); lens = np.diff(np.r_[starts, n_rows])
log(f"Runs: {len(starts)} | window = {W} samples, step = {S}")
keep = lens >= W
rs, rl = starts[keep], lens[keep]
nwin = (rl - W) // S + 1
tot = int(nwin.sum())
run_of_win = np.repeat(np.arange(len(rs)), nwin)
offs = np.arange(tot) - np.repeat(np.cumsum(nwin) - nwin, nwin)
w_start = (rs[run_of_win] + offs * S).astype(np.int64)
w_end = w_start + W - 1
# purity checks: first and last row of every window share subject / position / session / label
for nm, arr in [("subject", subj_c), ("position", pos_c), ("session", sess_c), ("label", lab_c)]:
    assert (arr[w_start] == arr[w_end]).all(), f"window crosses a {nm} boundary"
widx = pd.DataFrame({"start_row": w_start, "subject": [subjects[i] for i in subj_c[w_start]],
                     "position": [positions_all[i] for i in pos_c[w_start]],
                     "session": [sess_vals[i] for i in sess_c[w_start]],
                     "label": [labels[i] for i in lab_c[w_start]], "label_id": lab_c[w_start].astype(np.int16),
                     "run_id": rs[run_of_win]})
log("Windows:", len(widx), "| rows dropped (runs shorter than window):", int(lens[~keep].sum()))
try: widx.to_parquet(SHARED / "windows_index.parquet", index=False)
except Exception: widx.to_csv(SHARED / "windows_index.csv.gz", index=False)
json.dump({"window_samples": W, "step": S, "fs": FS, "labels": labels, "positions": positions_all, "subjects": subjects,
           "sessions": sess_vals, "data_sha256": DATA_SHA256}, open(SHARED / "windows_meta.json", "w"), indent=2)
display(widx.groupby(["position", "label"]).size().unstack("position"))

## 4 · Feature extraction (per channel, per window)
Only per-channel features: nothing mixes axes. A unit test confirms that features for a channel subset are identical to the matching columns of the full 6-channel matrix.

In [ ]:
FEATS = ["mean", "std", "min", "max", "median", "iqr", "rms", "skew", "kurt", "mad", "zcr",
         "dom_freq", "spec_energy", "spec_entropy", "spec_centroid"]

def window_features(X, ch_names, fs=FS):
    # X: (N, T, C) float -> (N, C*len(FEATS)) float32, names 'Ax__mean', ...
    N, T, C = X.shape
    freqs = np.fft.rfftfreq(T, d=1.0 / fs)[1:]
    out, names = [], []
    for c, ch in enumerate(ch_names):
        x = X[:, :, c].astype(np.float64)
        mu = x.mean(1); sd = x.std(1); xc = x - mu[:, None]
        q25, med, q75 = np.percentile(x, [25, 50, 75], axis=1)
        z = xc / np.where(sd > 1e-12, sd, 1.0)[:, None]
        P = np.abs(np.fft.rfft(xc, axis=1))[:, 1:] ** 2
        tot_p = P.sum(1); p = P / np.where(tot_p > 1e-12, tot_p, 1.0)[:, None]
        cols = [mu, sd, x.min(1), x.max(1), med, q75 - q25, np.sqrt((x ** 2).mean(1)),
                (z ** 3).mean(1), (z ** 4).mean(1) - 3.0, np.abs(np.diff(x, axis=1)).mean(1),
                (np.diff(np.signbit(xc).astype(np.int8), axis=1) != 0).mean(1),
                freqs[P.argmax(1)], tot_p / T, -(p * np.log(p + 1e-12)).sum(1), (p * freqs[None, :]).sum(1)]
        out += cols; names += [f"{ch}__{f}" for f in FEATS]
    return np.stack(out, axis=1).astype(np.float32), names

def gather(rows):                       # raw windows for given start rows -> (n, W, 6)
    return sig[rows[:, None] + np.arange(W)[None, :]]

# ---- unit test: subset features == matching columns of full features (no cross-channel dependence)
_rng = np.random.default_rng(SEED); _rows = w_start[_rng.choice(len(w_start), size=min(300, len(w_start)), replace=False)]
_Xw = gather(_rows)
_full, _names = window_features(_Xw, IMU)
for _sub in [["Ax"], ["Gy", "Gz"], ["Ax", "Ay", "Az"]]:
    _f, _n = window_features(_Xw[:, :, [IMU.index(c) for c in _sub]], _sub)
    _cols = [_names.index(n) for n in _n]
    assert np.allclose(_f, _full[:, _cols], atol=1e-4, rtol=1e-4), f"feature leakage across channels for {_sub}"
assert np.isfinite(_full).all()
log("Feature unit tests passed.", len(FEATS), "features per channel.")

POS_LIST = POSITIONS or positions_all
POS_DATA = {}
t0 = time.time()
for p in POS_LIST:
    sel = np.flatnonzero(widx["position"].to_numpy() == p)
    Xp = gather(w_start[sel])
    F, fnames = window_features(Xp, IMU)
    POS_DATA[p] = dict(sel=sel, F=F, names=fnames, y=widx["label_id"].to_numpy()[sel].astype(int),
                       subj=widx["subject"].to_numpy()[sel], starts=w_start[sel])
    log(f"{p}: windows={len(sel)}, feature matrix={F.shape}")
    del Xp
log("Features computed in", round(time.time() - t0, 1), "s")

## 5 · Axis configurations and models

In [ ]:
acc, gyr = ["Ax", "Ay", "Az"], ["Gx", "Gy", "Gz"]
ALL_CONFIGS = {"6-axis": IMU, "acc-3": acc, "gyro-3": gyr}
for a, b in itertools.combinations(acc, 2): ALL_CONFIGS[f"acc-{a}{b}"] = [a, b]
for a, b in itertools.combinations(gyr, 2): ALL_CONFIGS[f"gyro-{a}{b}"] = [a, b]
for k in IMU: ALL_CONFIGS[f"single-{k}"] = [k]
CONFIGS = {k: v for k, v in ALL_CONFIGS.items() if (CONFIG_NAMES is None or k in CONFIG_NAMES)}
FOLD_DEFS = [{"fold": i, "test_subject": s, "train_subjects": [x for x in subjects if x != s]} for i, s in enumerate(subjects)]
FOLD_LIST = [f for f in FOLD_DEFS if (FOLDS is None or f["fold"] in FOLDS)]
assert len(ALL_CONFIGS) == 15
log("Configs:", list(CONFIGS), "| folds:", [f["fold"] for f in FOLD_LIST], "| positions:", POS_LIST, "| models:", MODELS)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
try:
    import xgboost as xgb
except ImportError:
    os.system(f"{sys.executable} -m pip install -q xgboost"); import xgboost as xgb
import joblib
HAS_GPU = shutil.which("nvidia-smi") is not None

def balanced_weights(y, K):
    cnt = np.bincount(y, minlength=K).astype(float)
    w = np.where(cnt > 0, len(y) / (K * np.maximum(cnt, 1)), 0.0)
    return w[y]

def fit_model(name, Xtr, ytr, fold):
    # returns (model, scaler_or_None, info dict)
    if name == "RF":
        m = RandomForestClassifier(**RF_PARAMS).fit(Xtr, ytr)
        return m, None
    if name == "SVM":
        idx = np.arange(len(ytr))
        if len(idx) > SVM_MAX_TRAIN_WINDOWS:
            idx, _ = train_test_split(idx, train_size=SVM_MAX_TRAIN_WINDOWS, stratify=ytr, random_state=SEED + fold)
        sc = StandardScaler().fit(Xtr[idx])
        m = SVC(**SVM_PARAMS).fit(sc.transform(Xtr[idx]), ytr[idx])
        return m, sc
    if name == "XGB":
        params = dict(XGB_PARAMS)
        if HAS_GPU: params["device"] = "cuda"
        # all K classes are present in training (asserted in the loop), so ids 0..K-1 are contiguous
        m = xgb.XGBClassifier(objective="multi:softprob", **params)
        m.fit(Xtr, ytr, sample_weight=balanced_weights(ytr, K))
        return m, None
    raise ValueError(name)

def predict(name, m, sc, X):
    Z = sc.transform(X) if sc is not None else X
    return m.predict(Z)

def complexity(name, m, n_feat):
    buf = io.BytesIO(); joblib.dump(m, buf, compress=0); size = buf.tell()
    if name == "RF":
        nodes = int(sum(e.tree_.node_count for e in m.estimators_)); return size, nodes, len(m.estimators_)
    if name == "XGB":
        dump = m.get_booster().get_dump(); return size, int(sum(d.count("\n") for d in dump)), len(dump)
    if name == "SVM":
        nsv = int(m.n_support_.sum()); return size, int(nsv * n_feat + m.dual_coef_.size + m.intercept_.size), nsv

## 6 · Run integrity (config hash)

In [ ]:
design = {"window_samples": W, "step": S, "fs": FS, "features": FEATS, "labels": labels, "positions": POS_LIST,
          "subjects": subjects, "rf": RF_PARAMS, "svm": SVM_PARAMS, "svm_cap": SVM_MAX_TRAIN_WINDOWS, "xgb": XGB_PARAMS,
          "data_sha256": DATA_SHA256, "smoke": SMOKE_TEST, "seed": SEED}
cfg_hash = hashlib.sha256(json.dumps(design, sort_keys=True, default=str).encode()).hexdigest()
cfg_path = RUN_DIR / "run_config.json"
if cfg_path.exists():
    old = json.load(open(cfg_path))
    if old.get("hash") != cfg_hash and not FORCE_NEW_RUN:
        raise RuntimeError("Existing results were produced with DIFFERENT settings. Use a new RESULTS_ROOT, or set FORCE_NEW_RUN=True "
                           "after deleting results.csv / confusion_long.csv yourself.")
json.dump({"hash": cfg_hash, "design": design}, open(cfg_path, "w"), indent=2, default=str)
RES_PATH, CONF_PATH = RUN_DIR / "results.csv", RUN_DIR / "confusion_long.csv"
done = set()
if RES_PATH.exists():
    _d = pd.read_csv(RES_PATH, usecols=["position", "config", "model", "fold"])
    done = set(map(tuple, _d.to_numpy().tolist()))
log("Units already completed:", len(done))

## 7 · Main LOSO loop (resumable)

In [ ]:
workout_ids = [i for i in range(K) if i not in null_ids]
total_units = len(POS_LIST) * len(FOLD_LIST) * len(CONFIGS) * len(MODELS)
unit_i, t_start, n_new = 0, time.time(), 0
rows_buf, conf_buf = [], []

def flush():
    global rows_buf, conf_buf
    if rows_buf:
        pd.DataFrame(rows_buf).to_csv(RES_PATH, mode="a", header=not RES_PATH.exists(), index=False); rows_buf = []
    if conf_buf:
        pd.DataFrame(conf_buf).to_csv(CONF_PATH, mode="a", header=not CONF_PATH.exists(), index=False); conf_buf = []

for p in POS_LIST:
    D = POS_DATA[p]
    for fd in FOLD_LIST:
        test_s = fd["test_subject"]; te = D["subj"] == test_s; tr = ~te
        # ---- leakage guards
        assert test_s not in set(D["subj"][tr]), "test subject present in training windows"
        assert set(D["subj"][tr]) == set(fd["train_subjects"]), "training subjects differ from fold definition"
        ytr, yte = D["y"][tr], D["y"][te]
        Ftr_all, Fte_all = D["F"][tr], D["F"][te]
        assert len(np.unique(ytr)) == K, "a class is missing from the training subjects"
        present = np.unique(yte)
        for cname, ch in CONFIGS.items():
            cols = [i for i, n in enumerate(D["names"]) if n.split("__")[0] in ch]
            assert len(cols) == len(FEATS) * len(ch)
            Xtr, Xte = Ftr_all[:, cols], Fte_all[:, cols]
            for mname in MODELS:
                unit_i += 1
                key = (p, cname, mname, fd["fold"])
                if key in done: continue
                t0 = time.time()
                model, sc = fit_model(mname, Xtr, ytr, fd["fold"]); t_fit = time.time() - t0
                t0 = time.time(); pred = predict(mname, model, sc, Xte); t_pred = time.time() - t0
                cm = confusion_matrix(yte, pred, labels=np.arange(K))
                f1c = f1_score(yte, pred, labels=np.arange(K), average=None, zero_division=0)
                pw = [i for i in workout_ids if i in present]
                row = dict(position=p, config=cname, model=mname, fold=fd["fold"], test_subject=test_s, n_channels=len(ch),
                           channels="".join(ch), n_features=len(cols), n_train_windows=int(tr.sum()), n_test_windows=int(te.sum()),
                           macro_f1=float(f1_score(yte, pred, labels=present, average="macro", zero_division=0)),
                           macro_f1_workouts=float(f1c[pw].mean()) if pw else np.nan,
                           accuracy=float(accuracy_score(yte, pred)),
                           weighted_f1=float(f1_score(yte, pred, labels=present, average="weighted", zero_division=0)),
                           macro_precision=float(precision_score(yte, pred, labels=present, average="macro", zero_division=0)),
                           macro_recall=float(recall_score(yte, pred, labels=present, average="macro", zero_division=0)),
                           train_time_s=t_fit, predict_time_batch_s=t_pred)
                for i, l in enumerate(labels): row[f"f1_{l}"] = float(f1c[i]) if i in present else np.nan
                size, n_par, n_units = complexity(mname, model, len(cols))
                row.update(model_size_bytes=size, n_params_proxy=n_par, n_trees_or_sv=n_units)
                # ---- single-window latency (CPU, batch size 1): feature extraction + scaling + predict
                try:
                    if mname == "RF": model.set_params(n_jobs=1)
                    if mname == "XGB": model.set_params(n_jobs=1, device="cpu")
                except Exception: pass
                te_rows = D["starts"][te][:LATENCY_N_WINDOWS]
                chi = [IMU.index(c) for c in ch]; lf, lp = [], []
                for r0 in te_rows:
                    raw_w = sig[r0:r0 + W][None, :, chi]
                    a = time.perf_counter(); fx, _ = window_features(raw_w, ch); b = time.perf_counter()
                    _ = predict(mname, model, sc, fx); c = time.perf_counter()
                    lf.append(b - a); lp.append(c - b)
                row.update(latency_feat_ms=1000 * float(np.median(lf)), latency_predict_ms=1000 * float(np.median(lp)),
                           latency_total_ms=1000 * float(np.median(lf) + np.median(lp)))
                rows_buf.append(row)
                for i in range(K):
                    for j in range(K):
                        if cm[i, j]: conf_buf.append(dict(position=p, config=cname, model=mname, fold=fd["fold"], true=labels[i], pred=labels[j], n=int(cm[i, j])))
                done.add(key); n_new += 1
                if len(rows_buf) >= 10: flush()
                if n_new % 25 == 0 or n_new == 1:
                    el = time.time() - t_start; remaining = total_units - len(done)
                    log(f"[{len(done)}/{total_units}] {p} fold={fd['fold']} {cname} {mname}: macroF1={row['macro_f1']:.3f} "
                        f"fit={t_fit:.1f}s | elapsed {el/60:.1f} min, ETA ~ {el / n_new * remaining / 60:.1f} min")
    flush()
flush()
log("Loop finished. New units this session:", n_new, "| total done:", len(done), "of", total_units)

## 8 · Completeness check and descriptive summary
This is descriptive only. Statistical comparisons are done in the analysis notebook.

In [ ]:
res = pd.read_csv(RES_PATH)
exp = len(POS_LIST) * len(FOLD_LIST) * len(CONFIGS) * len(MODELS)
log("Rows in results.csv:", len(res), "| expected for this grid:", exp)
dups = res.duplicated(subset=["position", "config", "model", "fold"]).sum()
assert dups == 0, f"{dups} duplicated units in results.csv"
if len(res) < exp: log("INCOMPLETE: re-run the notebook to finish the remaining units.")

summ = (res.groupby(["position", "model", "config"])
          .agg(n_folds=("fold", "count"), macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
               macro_f1_workouts_mean=("macro_f1_workouts", "mean"), accuracy_mean=("accuracy", "mean"),
               weighted_f1_mean=("weighted_f1", "mean"), n_channels=("n_channels", "first"),
               params_proxy=("n_params_proxy", "mean"), size_kb=("model_size_bytes", lambda x: x.mean() / 1024),
               latency_ms=("latency_total_ms", "mean"))
          .reset_index())
summ.to_csv(RUN_DIR / "summary_by_position_model_config.csv", index=False)
display(summ.round(4).head(30))

## 9 · Environment record and zip

In [ ]:
import importlib
def ver(m):
    try: return importlib.import_module(m).__version__
    except Exception: return "n/a"
env = {"python": sys.version, "platform": platform.platform(), "gpu_visible": HAS_GPU, "seed": SEED, "smoke_test": SMOKE_TEST,
       "packages": {m: ver(m) for m in ["numpy", "pandas", "sklearn", "xgboost", "joblib"]},
       "data_sha256": DATA_SHA256, "config_hash": cfg_hash, "timestamp_utc": pd.Timestamp.now("UTC").isoformat()}
json.dump(env, open(RUN_DIR / "environment.json", "w"), indent=2)
(RUN_DIR / "run_log.txt").write_text("\n".join(LOG))

zip_path = BASE / f"recgym_01_classical_{'smoke' if SMOKE_TEST else 'full'}.zip"
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in RUN_DIR.rglob("*"):
        if f.is_file(): z.write(f, f"01_classical_{'smoke' if SMOKE_TEST else 'full'}/{f.relative_to(RUN_DIR)}")
    for f in SHARED.rglob("*"):
        if f.is_file(): z.write(f, f"shared/{f.name}")
log("ZIP:", zip_path, "| MB:", round(os.path.getsize(zip_path) / 1e6, 2))
if SMOKE_TEST:
    log("Smoke run finished. Check timing/ETA above, then set SMOKE_TEST = False for the full grid.")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    log("Not on Colab: find the zip in your working directory / Kaggle Output tab.")